## Preliminaries

You can run the notebook in two ways:

1. **Google Colab**: place the project folder `heat-forecast` in **MyDrive**. The setup cell below will mount Drive and automatically add `MyDrive/heat-forecast/src` to `sys.path`, so `import heat_forecast` works out of the box.

2. **Local machine**:

   * **Installing the package:** from the project root, run `pip install .` once. This installs the package normally, so you can open the notebook anywhere and import `heat_forecast` without modifying `sys.path`.
   * **Alternative:** if you are running the notebook directly from `.../heat-forecast/notebooks/` without installing the package, the setup cell will detect `../src` and automatically add it to `sys.path`.

In [ ]:
# --- Detect if running on Google Colab & Set base dir ---
# %cd /home/giovanni.lombardi/heat-forecast/notebooks
import subprocess
from pathlib import Path
import sys

def in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

# Install required packages only if not already installed
def pip_install(pkg: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Set base directory and handle environment
if in_colab():
    # Make sure IPython is modern (avoids the old %autoreload/imp issue if you ever use it)
    pip_install("ipython>=8.25")
    pip_install("ipykernel>=6.29")
    
    def install(package):
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

    for pkg in ["statsmodels", "statsforecast", "mlforecast"]:
        pip_install(pkg)

    # Mount Google Drive
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')

    # Set base directory to your Drive project folder
    BASE_DIR = Path('/content/drive/MyDrive/heat-forecast')

    # Add `src/` to sys.path for custom package imports
    SRC_PATH = BASE_DIR / 'src'
    if str(SRC_PATH) not in sys.path:
        sys.path.append(str(SRC_PATH))

    # Sanity checks (helpful error messages if path is wrong)
    assert SRC_PATH.exists(), f"Expected '{SRC_PATH}' to exist. Fix BASE_DIR."
    pkg_dir = SRC_PATH / "heat_forecast"
    assert pkg_dir.exists(), f"Expected '{pkg_dir}' package directory."
    init_file = pkg_dir / "__init__.py"
    assert init_file.exists(), f"Missing '{init_file}'. Add it so Python treats this as a package."

else:
    # Local: either rely on editable install (pip install -e .) or add src/ when running from repo
    # Assume notebook lives in PROJECT_ROOT/notebooks/
    BASE_DIR = Path.cwd().resolve().parent
    SRC_PATH = BASE_DIR / "src"

    added_src = False
    if (SRC_PATH / "heat_forecast").exists() and str(SRC_PATH) not in sys.path:
        sys.path.append(str(SRC_PATH))
        added_src = True

# --- Logging setup ---
import logging
from zoneinfo import ZoneInfo
from datetime import datetime

LOG_DIR  = (BASE_DIR / "logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / "run.log"
PREV_LOG = LOG_DIR / "run.prev.log"

# If there's a previous run.log with content, archive it to run.prev.log
if LOG_FILE.exists() and LOG_FILE.stat().st_size > 0:
    try:
        # Replace old run.prev.log if present
        if PREV_LOG.exists():
            PREV_LOG.unlink()
        LOG_FILE.rename(PREV_LOG)
    except Exception as e:
        # Fall back to truncating if rename fails (e.g., file locked)
        print(f"[warn] Could not archive previous log: {e}. Truncating current run.log.")
        LOG_FILE.write_text("")

# Configure logging: fresh file for this run + echo to notebook/stdout
file_handler   = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
stream_handler = logging.StreamHandler(sys.stdout)

fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s",
                        datefmt="%m-%d %H:%M:%S")
file_handler.setFormatter(fmt)
stream_handler.setFormatter(fmt)

root = logging.getLogger()
root.handlers[:] = [file_handler, stream_handler]  # replace handlers (important in notebooks)
root.setLevel(logging.INFO)

# Use Rome time
logging.Formatter.converter = lambda *args: datetime.now(ZoneInfo("Europe/Rome")).timetuple()

logging.captureWarnings(True)
logging.info("=== Logging started (fresh current run) ===")
logging.info("Previous run (if any): %s", PREV_LOG if PREV_LOG.exists() else "none")

if added_src:
    logging.info("heat_forecast not installed; added src/ to sys.path")
else:
    logging.info("heat_forecast imported without modifying sys.path (likely installed)")

OPTUNA_DIR = BASE_DIR / "results" / "tuning" / "lstm"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)
logging.info("BASE_DIR (make sure it's '*/heat-forecast/', else cd and re-run): %s", BASE_DIR)
logging.info("LOG_DIR: %s", LOG_DIR)
logging.info("OPTUNA_DIR: %s", OPTUNA_DIR)

Imports:

In [ ]:
# --- Magic Commands ---
%load_ext autoreload
%autoreload 2

# --- Standard Library ---
import os
#os.environ["OPTUNA_LOGGING_DISABLE_DEFAULT_HANDLER"] = "1" # prevent Optuna from attaching its handler
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Use only 1 GPU if available
import logging
from datetime import datetime
from itertools import product
import torch
from typing import Optional, List, Dict, Any, Union
#import optuna

# --- Third-Party Libraries ---
import numpy as np
import pandas as pd
pd.set_option('display.float_format', '{:.3f}'.format)

#import yaml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from tqdm.notebook import tqdm
from IPython.display import display, HTML

# --- Plotting Configuration ---
plt.style.use("seaborn-v0_8")
plt.rcParams['font.size'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.titlesize'] = 18
mpl.rcParams['axes.grid'] = True
mpl.rcParams['axes.grid.which'] = 'both'

# --- YAML Customization ---
from heat_forecast.utils.yaml import safe_dump_yaml

# --- Safe File Deletion Helper ---
from heat_forecast.utils.fileshandling import remove_tree

# --- Project-Specific Imports ---
from heat_forecast.utils.cv_utils import get_cv_params_for_test

# from heat_forecast.utils.optuna import (
#     OptunaStudyConfig, run_study, continue_study, describe_suggester, rename_study, clone_filtered_study
# )

logging.info("All imports successful.")

Import Chronos:

In [ ]:
# --- Chronos2 ---
# %pip install chronos-forecasting>=2.0 pandas[pyarrow] matplotlib
from chronos import BaseChronosPipeline, Chronos2Pipeline

Import pre-elaborated data:

In [ ]:
heat_path = BASE_DIR / 'data' / 'timeseries_preprocessed' / 'heat.csv'
aux_path = BASE_DIR / 'data' / 'timeseries_preprocessed' / 'auxiliary.csv'
heat_df = pd.read_csv(heat_path, parse_dates=['ds'])
aux_df = pd.read_csv(aux_path, parse_dates=['ds'])
logging.info("Loaded heat data: %s", heat_path.relative_to(BASE_DIR))
logging.info("Loaded auxiliary data: %s", aux_path.relative_to(BASE_DIR))

Set device:

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"Using device: {DEVICE}")

Set a seed (set to None to skip):

In [ ]:
SEED = 42
logging.info(f"Using seed: {SEED}")

Utils to add calendar features.

In [ ]:
from typing import Sequence

all_df = pd.merge(
    heat_df,
    aux_df,
    on=['unique_id', 'ds'],
    how='inner'
)

def add_calendar_features(
    df: pd.DataFrame,
    timestamp_column: str,
    features: Sequence[str] = ("hod", "dow", "moy", "wss"),
) -> pd.DataFrame:
    allowed = {"hod", "dow", "moy", "wss"}

    # Validate requested features
    if not isinstance(features, Sequence) or not all(isinstance(f, str) for f in features):
        raise ValueError("features must be a of strings")

    if not set(features).issubset(allowed):
        raise ValueError(f"features must be a subset of {sorted(allowed)}")

    tv = set(features)

    idx = pd.to_datetime(df[timestamp_column])
    dt_index = pd.DatetimeIndex(idx)

    need_hod = "hod" in tv
    need_dow = ("dow" in tv) or ("wss" in tv) # wss derived from dow
    need_moy = "moy" in tv

    if need_hod:
        hour = dt_index.hour.values
    if need_dow:
        dow = dt_index.dayofweek.values
    if need_moy:
        month = dt_index.month.values

    feats_dict: dict[str, np.ndarray] = {}

    def sincos(x: np.ndarray, period: int) -> tuple[np.ndarray, np.ndarray]:
        x = np.asarray(x, dtype=np.float32)
        ang = 2.0 * np.pi * (x / period)
        return np.sin(ang).astype(np.float32), np.cos(ang).astype(np.float32)

    if "hod" in tv:
        hour_sin, hour_cos = sincos(hour, 24)
        feats_dict["hod_sin"] = hour_sin
        feats_dict["hod_cos"] = hour_cos

    if "dow" in tv:
        dow_sin, dow_cos = sincos(dow, 7)
        feats_dict["dow_sin"] = dow_sin
        feats_dict["dow_cos"] = dow_cos

    if "moy" in tv:
        month_sin, month_cos = sincos(month - 1, 12)
        feats_dict["moy_sin"] = month_sin
        feats_dict["moy_cos"] = month_cos

    if "wss" in tv:
        wss = np.where(dow == 5, 1, np.where(dow == 6, 2, 0)).astype(np.int32)
        wss_sin, wss_cos = sincos(wss, 3)
        feats_dict["wss_sin"] = wss_sin
        feats_dict["wss_cos"] = wss_cos

    if feats_dict:
        feats_df = pd.DataFrame(feats_dict, index=df.index).astype(np.float32)
        df = pd.concat([df, feats_df], axis=1)

    return df

if not any(f in ['hod_sin', 'dow_sin', 'moy_sin', 'wss_sin'] for f in all_df.columns):
    all_df = add_calendar_features(all_df, 'ds', features=['hod', 'dow', 'moy', 'wss'])
# Limit to temperature and calendar features only
all_df_original = all_df.copy()
all_exog = ['temperature', 'dew_point', 'pressure', 'humidity', 'wind_speed']
exog_to_include = ['temperature']
exog_to_exclude = [e for e in all_exog if e not in exog_to_include]
all_df = all_df.drop(columns=exog_to_exclude, errors='ignore')
known_covariates = [col for col in ['hod_sin', 'hod_cos', 'dow_sin', 'dow_cos', 'moy_sin', 'moy_cos', 'wss_sin', 'wss_cos'] if col in all_df.columns] + exog_to_include

## Finetuning

In [ ]:
from dataclasses import dataclass, field, asdict

@dataclass
class FinetuningConfig:
    num_steps: int = 50
    learning_rate: float = 1e-5
    batch_size: int = 32
    logging_steps: int = 10


def cross_validation(
    finetuning_config: FinetuningConfig,
    test_size: int,
    prediction_length: int,
    all_id_df: pd.DataFrame,
    end_test: Optional[pd.Timestamp] = None,
    step_size: int = 1,
    input_size: Optional[int] = None,  
    refit: Union[bool, int] = True,     
    verbose: bool = True,
    alias: str = "Chronos-2",
    device: Union[str, torch.device] = "auto",
    known_covariates: list[str] = ['hod_sin', 'hod_cos', 'dow_sin', 'dow_cos', 'moy_sin', 'moy_cos', 'wss_sin', 'wss_cos', 'temperature'],
    target: str = 'y',
    timestamp_column: str = 'ds',
    id_column: str = 'id',
) -> pd.DataFrame:

    h = prediction_length
    if h <= 0 or test_size <= 0 or step_size <= 0:
        raise ValueError("h, test_size, and step_size must be positive integers.")
    if (test_size - h) % step_size != 0:
        raise ValueError("`test_size - h` must be a multiple of `step_size`.")

    # Build relative offsets for window cutoffs
    steps = list(range(-test_size, -h + 1, step_size))  # last cutoff is end_test - h

    # Determine end_test
    all_id_df.set_index(timestamp_column, inplace=True)
    idx_all = all_id_df.index  
    if end_test is None:
        end_test = idx_all.max()
    else:
        if not isinstance(end_test, pd.Timestamp):
            raise ValueError("end_test must be a pandas.Timestamp.")
        if end_test not in idx_all:
            raise ValueError(f"end_test ({end_test}) must be present in the prepared target index.")

    # Check if the given input size (amount of data for each train) is valid
    if input_size is not None:
        if not isinstance(input_size, int) or input_size <= 0:
            raise ValueError("input_size must be a positive integer (hours).")
        first_cutoff = end_test + pd.Timedelta(hours=steps[0])
        first_start_train = first_cutoff - pd.Timedelta(hours=input_size) + pd.Timedelta(hours=1)
        if first_start_train < self._y.index.min():
            raise ValueError(
                f"input_size ({input_size} hours) is too large for the given end test ({end_test}). "
                f"Training would start at {first_start_train}, but the earliest available data in .prepared_data is {self._y.index.min()}."
            )

    # iterate windows 
    all_results = []
    prev_pipeline: Optional["Chronos2Pipeline"] = None
    if isinstance(device, str):
        if not device in ["cpu", "cuda", "auto"]:
            raise ValueError("device string must be 'cpu', 'cuda', or 'auto'.")
        if device == "cuda" and not torch.cuda.is_available():
            raise ValueError("CUDA device requested but not available.")
        _DEVICE = torch.device(device) if device != "auto" else (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
    else:
        if not isinstance(device, torch.device):
            raise ValueError("device must be a string or torch.device.")
        _DEVICE = device
    
    iterator = tqdm(steps, disable=not verbose, desc="CV windows", leave=True)
    for i, offset in enumerate(iterator):
        cutoff = end_test + pd.Timedelta(hours=offset)

        # training slice for this window
        if input_size is not None:
            start_train = cutoff - pd.Timedelta(hours=input_size) + pd.Timedelta(hours=1)
        else:
            start_train = self._y.index.min()
        end_train = cutoff
        train_mask = (all_id_df.index >= start_train) & (all_id_df.index <= end_train)
        train_df = all_id_df.loc[train_mask].copy()

        # decide whether to refit
        do_refit = (
            i == 0
            or (isinstance(refit, int) and not isinstance(refit, bool) and i % int(refit) == 0)
            or (refit is True)
            or (refit is None)
        )

        if do_refit or prev_pipeline is None:
            # Build a fresh pipeline but reuse precomputed features to avoid recomputing
            pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=_DEVICE)
            train_inputs = [{
                "target": train_df[target].values,
                "past_covariates": {col: train_df[col].values for col in known_covariates}, # we havo no past-only covariates
                # Future values of covariates are not used during training.
                # However, we need to include their names to indicate that these columns will be available at prediction time
                "future_covariates": {col: None for col in known_covariates},
            }]
            fc = finetuning_config
            pipeline = pipeline.fit(
                inputs=train_inputs,
                prediction_length=prediction_length,
                num_steps=fc.num_steps,  # few fine-tuning steps for a quick demo
                learning_rate=fc.learning_rate,
                batch_size=fc.batch_size,
                logging_steps=fc.logging_steps,
            )
            prev_pipeline = pipeline
        else:
            pipeline = prev_pipeline

        past_df = all_id_df[all_id_df.index <= cutoff]
        future_df = all_id_df[(all_id_df.index > cutoff) &
                            (all_id_df.index <= cutoff + pd.Timedelta(hours=h))].drop(columns=[target])
        pred_df = pipeline.predict_df(
            past_df,
            future_df=future_df,
            prediction_length=h,
            quantile_levels=[],
            id_column=id_column,
            timestamp_column=timestamp_column,
            target=target,
        ).drop(columns=['target_name'])
        pred_df['cutoff'] = cutoff
        all_results.append(pred_df)

    result = pd.concat(all_results, ignore_index=True)
    return result.rename(columns={"predictions": alias})


In [ ]:
finetune_chronos = False   # Set to False to skip fine tuning

horizon_type = "day"      # "day" or "week"
unique_id = "F1"          # single ID for fine tuning

# Grid over num_steps
num_steps_grid = [10, 50, 100, 150, 200, 250]
lr_grid = [1e-4, 1e-5]
lr_name_grid = ["1em4", "1em5"]

# Chronos CV settings that you keep fixed across the grid
n_fits = 5
chronos_settings = dict(
    prediction_length = 24 if horizon_type == "day" else 24 * 7,
    step_size         = 24,
    batch_size        = 64,         
    logging_steps     = 50,
    refit             = 37,  
    test_size         = 4416,   
    end_test          = pd.Timestamp("2024-04-15 23:00:00"),       
    input_size        = None,    
    device            = "cuda",   
    known_covariates  = ["hod_sin", "hod_cos",
                         "dow_sin", "dow_cos",
                         "moy_sin", "moy_cos",
                         "wss_sin", "wss_cos",
                         "temperature"],
    target            = "y",
    timestamp_column  = "ds",
    id_column         = "unique_id",
)

# === Do not edit below ===
if finetune_chronos:
    metadata = {}
    cv_frames: list[pd.DataFrame] = []
    records: list[dict] = []

    if horizon_type not in ["day", "week"]:
        raise ValueError("Unsupported horizon type. Use 'day' or 'week'.")

    # horizon in hours
    h = 24 if horizon_type == "day" else 24 * 7

    # filter to single series
    heat_facilities_df = heat_df[heat_df["unique_id"] == unique_id].copy()

    # --- Create directory for fine tuning results ---
    timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    run_id = f"{unique_id}_{horizon_type}_finetuning_chronos2_{timestamp}"
    path = BASE_DIR / "results" / "tuning" / "chronos2" / run_id

    metadata["run_id"] = run_id
    metadata["horizon_type"] = horizon_type
    metadata["horizon_hours"] = h
    metadata["num_steps_grid"] = num_steps_grid
    metadata["chronos_settings"] = {k: str(v) for k, v in chronos_settings.items()}

    try:
        path.mkdir(parents=True, exist_ok=False)
        logging.info(
            "Created directory for Chronos-2 fine tuning results: %s",
            path.relative_to(BASE_DIR),
        )

        # Run grid search for Chronos-2 model
        t_strt_cv = pd.Timestamp.now()

        for i, (n_steps_idx, lr_idx) in enumerate(
            tqdm(product(range(len(num_steps_grid)), range(len(lr_grid))), desc="Chronos-2 grid", leave=True)
        ):
            # build FinetuningConfig for this value of num_steps
            n_steps = num_steps_grid[n_steps_idx]
            lr = lr_grid[lr_idx]
            fc = FinetuningConfig(
                num_steps     = n_steps,
                learning_rate = lr,
                batch_size    = chronos_settings["batch_size"],
                logging_steps = chronos_settings["logging_steps"],
            )

            alias = f"Ch2_s{n_steps}_lr{lr_name_grid[lr_idx]}"

            t0 = pd.Timestamp.now()
            cv_chronos = cross_validation(
                finetuning_config  = fc,
                test_size          = chronos_settings["test_size"],
                prediction_length  = chronos_settings["prediction_length"],
                all_id_df          = heat_facilities_df,
                end_test           = chronos_settings["end_test"],
                step_size          = chronos_settings["step_size"],
                input_size         = chronos_settings["input_size"],
                refit              = chronos_settings["refit"],
                verbose            = False,
                alias              = alias,
                device             = chronos_settings["device"],
                known_covariates   = chronos_settings["known_covariates"],
                target             = chronos_settings["target"],
                timestamp_column   = chronos_settings["timestamp_column"],
                id_column          = chronos_settings["id_column"],
            )
            t1 = pd.Timestamp.now()
            elapsed = (t1 - t0).total_seconds()

            cv_frames.append(cv_chronos)

            record = {
                "name": alias,
                "num_steps": n_steps,
                "learning_rate": lr,
                "batch_size": chronos_settings["batch_size"],
                "avg_elapsed_sec": elapsed / n_fits,
            }
            records.append(record)

        # Combine all cross validation results into a single DataFrame
        key_cols = [
            chronos_settings["id_column"],
            chronos_settings["timestamp_column"],
            "cutoff",
        ]

        cv_results_df = reduce(
            lambda left, right: pd.merge(
                left, right, on=key_cols, how="outer"
            ),
            cv_frames,
        )
        records_df = pd.DataFrame(records)
        results = (records_df, cv_results_df)

        t_end_cv = pd.Timestamp.now()
        elapsed_cv = str(t_end_cv - t_strt_cv)
        metadata["total_elapsed_time"] = elapsed_cv

        # --- Final save ---
        records_df.to_parquet(path / "records.parquet", compression="snappy")
        cv_results_df.to_parquet(path / "cv_results.parquet", compression="snappy")

        metadata_path = path / "metadata.yaml"
        with open(metadata_path, "w") as f:
            safe_dump_yaml(
                metadata,
                f,
                indent=4,
            )
        logging.info("Chronos-2 fine tuning completed successfully.")

    except KeyboardInterrupt:
        logging.warning("Interrupted; cleaning up Chronos-2 run.")
        remove_tree(path, require_within=BASE_DIR)
        logging.info(
            "Removed %s",
            path.relative_to(BASE_DIR)
            if BASE_DIR in path.resolve().parents
            else path,
        )
        raise
    except Exception:
        logging.exception(
            "Error during Chronos-2 test for id=%s, horizon=%s; cleaning up.",
            unique_id,
            horizon_type,
        )
        remove_tree(path, require_within=BASE_DIR)
        logging.info(
            "Removed %s",
            path.relative_to(BASE_DIR)
            if BASE_DIR in path.resolve().parents
            else path,
        )
        raise

    logging.info("Tuning completed.")

In [ ]:
load_results = False

if load_results:
    # run_id = ... 
    path = BASE_DIR / "results" / "tuning" / "chronos2" / run_id
    records_df = pd.read_parquet(path / "records.parquet")
    cv_results_df = pd.read_parquet(path / "cv_results.parquet")
    cv_results_df

## Validation of different settings

In [ ]:
target = "y"  # Column name containing the values to forecast (energy prices)
id_column = "unique_id"  # Column identifying different time series (countries/regions)
timestamp_column = "ds"  # Column containing datetime information

name_for_variant = "Chronos-2"
name_in_folder = "chron"
exog_vars = ['temperature']

exog_to_exclude = [e for e in all_exog if e not in exog_vars]
all_df = all_df_original.drop(columns=exog_to_exclude, errors='ignore')

In [ ]:
horizon_type = 'day' # 'day' or 'week'
id = 'F1'
run_test = True

if run_test:
    first_cutoff = pd.Timestamp("2024-10-01 23:00:00")
    if horizon_type == 'week':
        cutoffs = [first_cutoff + pd.Timedelta(weeks=i) for i in range(27)]
    elif horizon_type == 'day':
        cutoffs = [first_cutoff + pd.Timedelta(days=i) for i in range(183)]
    else:
        raise ValueError("horizon_type must be 'day' or 'week'")

    all_id_df = all_df[all_df[id_column] == id]
    h = 24 if horizon_type == 'day' else 168

    results = []
    pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cpu")
    for c in tqdm(cutoffs, desc="Forecasting"):
        past_df = all_id_df[all_id_df[timestamp_column] <= c]
        future_df = all_id_df[(all_id_df[timestamp_column] > c) &
                            (all_id_df[timestamp_column] <= c + pd.Timedelta(hours=h))].drop(columns=[target])
        pred_df = pipeline.predict_df(
            past_df,
            future_df=future_df,
            prediction_length=h,
            quantile_levels=[],
            id_column=id_column,
            timestamp_column=timestamp_column,
            target=target,
        ).drop(columns=['target_name'])
        pred_df['cutoff'] = c
        results.append(pred_df)

    predictions = pd.concat(results, ignore_index=True).rename(columns={'predictions': name_for_variant})
    predictions.head()

In [ ]:
save = True
if save and run_test:
    timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    run_id = f"{id}_{horizon_type}_val_chron_{timestamp}"
    path = BASE_DIR / "results" / "tuning" / "chron" / run_id
    path.mkdir(parents=True, exist_ok=False)
    predictions.to_parquet(path / "cv_df.parquet", compression="snappy")
    logging.info(f"Predictions successfully saved to: {path / 'cv_df.parquet'}")

In [ ]:
from functools import reduce

id = "F1"
horizon = "week"
id_horizon = id + "_" + horizon
path = BASE_DIR / "results" / "tuning" / "chron" / id_horizon

# Retrive results
dfs = []
for run_dir in path.iterdir():
    cv_path = run_dir / "cv_df.parquet"
    tmp = pd.read_parquet(cv_path)
    dfs.append(tmp)
df = reduce(lambda left, right: left.merge(right, on=["unique_id", "cutoff", "ds"], how="outer"), dfs)

# Add target aligned on ds
ds_vals = dfs[0]["ds"].unique()

t = heat_df.loc[
    (heat_df["unique_id"] == id)
    & (heat_df["ds"].isin(ds_vals)),
].copy()
df = df.merge(t, on=["unique_id", "ds"])


In [ ]:
from heat_forecast.utils.evaluation import evaluate_cv_forecasts, cv_evaluation_summary, display_cv_summary

combined_results = evaluate_cv_forecasts(
    cv_df=df,
    metrics=['mae', 'rmse', 'mase', 'nmae'], 
    target_df=heat_df, 
    period_for_nmae=(pd.Timestamp('2024-12-21'), pd.Timestamp('2025-03-20'))
)
summary = cv_evaluation_summary(combined_results)

In [ ]:
wide_summary = display_cv_summary(
    summary,
    sort_metric='mae',  
    sort_stat='mean',   # Sort by the mean of the metric
    ascending=True,     # Sort by ascending order of the metric
    by_panel=True,
    show_row_numbers=True
)

### Bootstrap

In [ ]:
from typing import Tuple, Iterable
import plotly.graph_objects as go
from heat_forecast.utils.plotting import add_season_background

all_results = evaluate_cv_forecasts(
    cv_df=df,
    target_df=heat_df, 
)

def compute_deltas(
    all_results: pd.DataFrame,
    stationarity_period: Tuple[pd.Timestamp, pd.Timestamp],
    metrics: Iterable[str],
    model_cols: Iterable[str],
    base_model: str = "LSTM",
    return_plot: bool = True,
):
    deltas_all_dict = {}
    deltas_stat_dict = {}
    for metric in metrics:
        # Select relevant models
        errors = all_results.loc[all_results['metric'] == metric, model_cols+['cutoff']].reset_index(drop=True)
        errors = errors[errors['cutoff']>pd.Timestamp('2024-08-10')].reset_index(drop=True)
        delta_cols = {
            col: errors[col] - errors[base_model]
            for col in model_cols
            if col != base_model
        }
        deltas_all = pd.DataFrame({
            'cutoff': errors['cutoff'],
            **delta_cols
        })
        deltas_all_dict[metric] = deltas_all
        # Select a period of approx stationarity
        deltas_stationary = deltas_all[
            deltas_all['cutoff'].between(stationarity_period[0], stationarity_period[1])
        ].reset_index(drop=True)
        deltas_stat_dict[metric] = deltas_stationary
    if return_plot:
        figs: list[go.Figure] = []
        for metric, deltas in deltas_all_dict.items():
            fig = go.Figure()
            delta_model_cols = set(model_cols).difference([base_model])
            for col in delta_model_cols:
                fig.add_trace(
                    go.Scatter(
                        x=deltas['cutoff'],
                        y=deltas[col],
                        mode='lines',
                        name=col
                    )
                )
            first_x, sec_x = stationarity_period[0], stationarity_period[1]
            fig.add_shape(
                type="line",
                x0=first_x, x1=first_x, y0=0, y1=1,
                xref="x", yref="paper",
                line=dict(color="red", width=1, dash="dash"),
            )
            fig.add_shape(
                type="line",
                x0=sec_x, x1=sec_x, y0=0, y1=1,
                xref="x", yref="paper",
                line=dict(color="red", width=1, dash="dash"),
            )
            add_season_background(fig, deltas['cutoff'].min(), deltas['cutoff'].max())
            fig.update_layout(title=f"{metric.upper()} deltas vs {base_model} (n° cutoffs = {len(deltas_stat_dict[metric])})",)
            figs.append(fig)
        return deltas_stat_dict, figs
    return deltas_stat_dict, []

deltas_dict, deltas_figs = compute_deltas(
    all_results=all_results,
    stationarity_period=(pd.Timestamp('2024-11-10'), pd.Timestamp('2025-03-17')),
    metrics=['mae', 'rmse', 'smape'],
    model_cols=['Chronos-2-AE', 'Chronos-2', 'Chronos-2-AE-ICR', 'Chronos-2-ICR'],
    base_model='Chronos-2-AE',
    return_plot=True
)
for fig in deltas_figs:
    display(fig)

import statsmodels.api as sm
from arch.unitroot import DFGLS, KPSS

metrics = list(deltas_dict.keys())
delta_model_cols = deltas_dict[metrics[0]].columns.difference(['cutoff']).tolist()
base_model = 'LSTM'
model_cols = delta_model_cols + [base_model]

max_lag = 40  

acf_figs = []

for metric in metrics:
    deltas = deltas_dict[metric]
    for col in delta_model_cols:
        print(f"\n--- Analysis for {metric.upper()} - {col} vs {base_model} ---")
        series = deltas[col].dropna().to_numpy()

        # unit root test
        dfg = DFGLS(series, trend="c", method="aic")
        print(f"DF-GLS pval: {dfg.pvalue:0.4f} \t (reject H0 -> stationary around mean)")
        kpss = KPSS(series, trend="c", lags=None)
        print(f"KPSS pval  : {kpss.pvalue:0.4f} \t (fail to reject H0 -> stationary around mean)")

        # acf
        acf_vals = sm.tsa.stattools.acf(series, nlags=max_lag, fft=True)

        # approximate 95 % confidence band for white noise
        n = len(series)
        conf = 1.96 / np.sqrt(n)

        fig = go.Figure()
        fig.add_trace(
            go.Bar(
                x=list(range(len(acf_vals))),
                y=acf_vals,
                name=f"ACF {metric.upper()} - {col} vs {base_model}"
            )
        )
        fig.add_hline(y=conf, line_width=1, line_dash="dash")
        fig.add_hline(y=-conf, line_width=1, line_dash="dash")

        fig.update_layout(
            title=f"ACF of stationary deltas - {metric.upper()} - {col} vs {base_model}",
            xaxis_title="Lag",
            yaxis_title="Autocorrelation"
        )
        acf_figs.append(fig)

# display all ACF figures (in Jupyter)
for fig in acf_figs:
    display(fig)


In [ ]:
from plotly.subplots import make_subplots
from recombinator.optimal_block_length import optimal_block_length
from recombinator.block_bootstrap import circular_block_bootstrap

import numpy as np

def holm_bonferroni(pvals: np.ndarray) -> np.ndarray:
    """
    Holm-Bonferroni adjusted p-values (step-down), controlling FWER.
    """
    p = np.asarray(pvals, dtype=float)
    m = p.size
    order = np.argsort(p)
    p_sorted = p[order]

    q = (m - np.arange(m)) * p_sorted  # (m-k+1) * p_(k)
    adj_sorted = np.empty(m)
    running_max = 0.0
    for i in range(m):
        val = min(q[i], 1.0)
        running_max = max(running_max, val)
        adj_sorted[i] = running_max

    p_adj = np.empty(m)
    p_adj[order] = adj_sorted
    return p_adj

def bootstrap_mean_delta(
    deltas_dict: Dict[str, pd.DataFrame],
    n_boot: int = 10_000,
    alpha: float = 0.05,
    n_cols: int = 2,
    width_per_col: int = 400,
    height_per_row: int = 320,
    block_length: Optional[int] = None,
) -> Tuple[Dict[str, pd.DataFrame], go.Figure]:
    results: Dict[str, pd.DataFrame] = {}

    # ---------- compute bootstrap stats per metric ----------
    for metric, deltas in deltas_dict.items():
        delta_cols = deltas.columns.difference(["cutoff"])
        results_metric = []

        for col in delta_cols:
            x = deltas[col].to_numpy()
            x = x[~np.isnan(x)]

            # Optimal block length for this series
            if block_length is not None:
                b_cb = block_length
            else:
                b_star = optimal_block_length(x)[0]
                b_cb = int(np.ceil(b_star.b_star_cb))

            # Circular block bootstrap
            samples = circular_block_bootstrap(
                x,
                block_length=b_cb,
                replications=n_boot,
                replace=True,
            )  # shape = (replications, len(x))

            boot_means = samples.mean(axis=1)

            # CI (percentile bootstrap)
            q_low, q_high = np.percentile(
                boot_means,
                [100 * alpha / 2, 100 * (1 - alpha / 2)],
            )

            # Two-sided p-value for H0: Delta = 0 vs H1: Delta != 0
            obs = x.mean()  # media dei valori osservati
            null_samples = boot_means - obs # boot_means sono ottenuti pescando con ripetizione dai valori osservati,
                                            # traslando stiamo "simulando la distribuzione nulla"
            p_two_sided = np.mean(np.abs(null_samples) >= np.abs(obs))

            results_metric.append(
                {
                    "model": col,
                    "mean_delta": x.mean(),
                    "ci_low": q_low,
                    "ci_high": q_high,
                    "block_length_cb": b_cb,
                    "p_raw": p_two_sided,
                }
            )

        # Collect everything in a DataFrame
        ci_df = pd.DataFrame(results_metric)

        # ---- Holm–Bonferroni correction (within this metric) ----
        p = ci_df["p_raw"].values
        p_holm = holm_bonferroni(p)
        ci_df["p_holm"] = p_holm
        results[metric] = ci_df

    # ---------- GLOBAL Holm across all metrics x models ----------
    all_rows = []
    for metric, df_m in results.items():
        tmp = df_m.copy()
        tmp["metric"] = metric
        all_rows.append(tmp)

    all_df = pd.concat(all_rows, ignore_index=True)

    all_df["p_holm_global"] = holm_bonferroni(all_df["p_raw"].values)

    # Write back into the per-metric results frames
    for metric in results.keys():
        mask = all_df["metric"] == metric
        # preserve original row order by merging on model
        p_map = dict(zip(all_df.loc[mask, "model"], all_df.loc[mask, "p_holm_global"]))
        results[metric]["p_holm_global"] = results[metric]["model"].map(p_map).to_numpy()

    # ---------- forest plots in subplots ----------
    metrics = list(results.keys())
    n_metrics = len(metrics)
    n_rows = int(np.ceil(n_metrics / n_cols))

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=[m.upper() for m in metrics],
        shared_yaxes=False,
    )

    for idx, metric in enumerate(metrics):
        row = idx // n_cols + 1
        col = idx % n_cols + 1

        ci_df = results[metric]
        plot_df = ci_df.sort_values("mean_delta").reset_index(drop=True)

        # Mean Δ points
        fig.add_trace(
            go.Scatter(
                x=plot_df["model"],
                y=plot_df["mean_delta"],
                mode="markers",
                marker=dict(size=8, color="black"),
                name="Mean Δ",
                showlegend=(idx == 0),
            ),
            row=row,
            col=col,
        )

        # CI bars (lower bound)
        fig.add_trace(
            go.Scatter(
                x=plot_df["model"],
                y=plot_df["ci_low"],
                mode="lines",
                line=dict(width=1, color="gray"),
                showlegend=False,
            ),
            row=row,
            col=col,
        )

        # CI bars (upper bound) + fill between
        fig.add_trace(
            go.Scatter(
                x=plot_df["model"],
                y=plot_df["ci_high"],
                mode="lines",
                line=dict(width=1, color="gray"),
                fill="tonexty",
                fillcolor="rgba(100,100,100,0.15)",
                name=f"{int((1-alpha)*100)}% CI" if idx == 0 else None,
                showlegend=(idx == 0),
            ),
            row=row,
            col=col,
        )

        # Reference line: "no improvement" (Delta = 0)
        fig.add_hline(
            y=0,
            line_width=1,
            line_dash="dash",
            row=row,
            col=col,
        )

    fig.update_layout(
        title="Mean Δ with Bootstrap 95% CI",
        xaxis_title="Model",
        yaxis_title="Mean Δ",
        template="simple_white",
        width=width_per_col * n_cols,
        height=height_per_row * n_rows,
    )

    return results, fig

results, fig = bootstrap_mean_delta(
    deltas_dict=deltas_dict,
    n_boot=10_000,
    alpha=0.05,
    n_cols=3,
    width_per_col=400,
    height_per_row=400,
)
boot_results_F1 = pd.concat(
    {metric: df.set_index("model") for metric, df in results.items()},
    axis=0
)
display(fig)
display(boot_results_F1)

## Testing: Univariate Forecasting with Covariates

In [ ]:
target = "y"  # Column name containing the values to forecast (energy prices)
id_column = "unique_id"  # Column identifying different time series (countries/regions)
timestamp_column = "ds"  # Column containing datetime information

In [ ]:
horizon_type = 'week' # 'day' or 'week'
id = 'F5'
run_test = True

if run_test:
    first_cutoff = pd.Timestamp("2024-05-19 23:00:00")
    if horizon_type == 'week':
        cutoffs = [first_cutoff + pd.Timedelta(weeks=i) for i in range(52)]
    elif horizon_type == 'day':
        cutoffs = [first_cutoff + pd.Timedelta(days=i) for i in range(365)]
    else:
        raise ValueError("horizon_type must be 'day' or 'week'")

    all_id_df = all_df[all_df[id_column] == id]
    h = 24 if horizon_type == 'day' else 168

    results = []
    pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cuda")
    for c in tqdm(cutoffs, desc="Forecasting"):
        past_df = all_id_df[all_id_df[timestamp_column] <= c]
        future_df = all_id_df[(all_id_df[timestamp_column] > c) &
                            (all_id_df[timestamp_column] <= c + pd.Timedelta(hours=h))].drop(columns=[target])
        pred_df = pipeline.predict_df(
            past_df,
            future_df=future_df,
            prediction_length=h,
            quantile_levels=[],
            id_column=id_column,
            timestamp_column=timestamp_column,
            target=target,
        ).drop(columns=['target_name'])
        pred_df['cutoff'] = c
        results.append(pred_df)

    predictions = pd.concat(results, ignore_index=True).rename(columns={'predictions': 'Chronos-2'})
    predictions.head()

In [ ]:
save = False
if save and run_test:
    timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    run_id = f"{id}_{horizon_type}_test_chron_{timestamp}"
    path = BASE_DIR / "results" / "test" / "chron" / run_id
    path.mkdir(parents=True, exist_ok=False)
    predictions.to_parquet(path / "cv_df.parquet", compression="snappy")
    logging.info(f"Predictions successfully saved to: {path / 'cv_df.parquet'}")

## Testing with cross-learning

In [ ]:
target = "y"  # Column name containing the values to forecast (energy prices)
id_column = "unique_id"  # Column identifying different time series (countries/regions)
timestamp_column = "ds"  # Column containing datetime information

In [ ]:
horizon_type = 'week' # 'day' or 'week'
run_test = True

if run_test:
    first_cutoff = pd.Timestamp("2024-05-19 23:00:00")
    if horizon_type == 'week':
        cutoffs = [first_cutoff + pd.Timedelta(weeks=i) for i in range(52)]
    elif horizon_type == 'day':
        cutoffs = [first_cutoff + pd.Timedelta(days=i) for i in range(365)]
    else:
        raise ValueError("horizon_type must be 'day' or 'week'")

    h = 24 if horizon_type == 'day' else 168

    results = []
    DEVICE = "cuda"
    pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)
    for c in tqdm(cutoffs, desc="Forecasting"):
        past_df = all_df[all_df[timestamp_column] <= c].copy()
        future_df = all_df[(all_df[timestamp_column] > c) &
                            (all_df[timestamp_column] <= c + pd.Timedelta(hours=h))].drop(columns=[target]).copy()
        pred_df = pipeline.predict_df(
            past_df,
            future_df=future_df,
            prediction_length=h,
            quantile_levels=[],
            id_column=id_column,
            timestamp_column=timestamp_column,
            target=target,
            predict_batches_jointly=True,
        ).drop(columns=['target_name'])
        pred_df['cutoff'] = c
        results.append(pred_df)

    predictions = pd.concat(results, ignore_index=True).rename(columns={'predictions': 'Chronos-2-ICR'})
    predictions.head()

In [ ]:
save = False
if save and run_test:
    for id in predictions['unique_id'].unique():
        id_df = predictions[predictions['unique_id'] == id]
        timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
        run_id = f"{id}_{horizon_type}_test_chronICR_{timestamp}"
        path = BASE_DIR / "results" / "test" / "chronICR" / run_id
        path.mkdir(parents=True, exist_ok=False)
        id_df.to_parquet(path / "cv_df.parquet", compression="snappy")
    logging.info(f"Predictions successfully.")

## Inference Times

In [ ]:
# Choose unique_id 
unique_id = 'F1'
horizon_type = 'day'
device_str = "cpu" # "cpu" or "cuda"
repeats = 10

timestamp_column = "ds"
target = "y"
id_column = "unique_id"

timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
run_id = f"{unique_id}_{horizon_type}_{device_str}_times_chron_{timestamp}"
logging.info(f"Run ID for timing: {run_id} (or pick an already existing one for resuming)")

In [ ]:
# Pick run_id
run_id = "F1_day_cpu_times_chron_20251204T162254"

# === Do not edit below ===
end_train = pd.Timestamp("2024-12-31T23:00:00")
h = 24 if horizon_type == 'day' else 168
all_id_df = all_df[all_df[id_column] == unique_id].copy()
past_df = all_id_df[all_id_df[timestamp_column] <= end_train]
future_df = all_id_df[(all_id_df[timestamp_column] > end_train) &
                    (all_id_df[timestamp_column] <= end_train + pd.Timedelta(hours=h))].drop(columns=[target])
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=device_str)

results_dir = BASE_DIR / "results" / "times" / "chron" / run_id
times_path = results_dir / "times.pkl"

logging.info(
    f"Retrieving directory for recording times: {times_path.relative_to(BASE_DIR)}"
)

os.makedirs(results_dir, exist_ok=True)
import pickle
if os.path.exists(times_path):
    try:
        with open(times_path, "rb") as f:
            data = pickle.load(f)
        if not isinstance(data, dict):
            data = {"train": [], "inference": []}
    except Exception:
        data = {"train": [], "inference": []}
else:
    data = {"train": [], "inference": []}

for _ in tqdm(
    range(max(0, repeats - len(data["inference"]))),
    desc="Repeat forecasts",
    leave=True,
):
    t0 = pd.Timestamp.now()
    pred_df = pipeline.predict_df(
        past_df,
        future_df=future_df,
        prediction_length=h,
        quantile_levels=[],
        id_column=id_column,
        timestamp_column=timestamp_column,
        target=target,
    )
    t1 = pd.Timestamp.now()

    elapsed = (t1 - t0).total_seconds()
    data["inference"].append(elapsed)

    # Persist
    with open(times_path, "wb") as f:
        pickle.dump(data, f)

logging.info(
    f"Timing data saved to: {times_path.relative_to(BASE_DIR)}"
)


In [ ]:
run_id = "F1_week_cuda_times_chron_20260304T074633"
n_warmup = 2  # number of initial runs to skip in stats
n = 8         # total number of runs used for stats

# === Do not edit below ===
results_dir = BASE_DIR / "results" / "times" / "chron" / run_id
times_path = results_dir / "times.pkl"
import pickle
with open(times_path, "rb") as f:
    data = pickle.load(f)
training_times = np.array(data.get("train", []))
inference_times = np.array(data.get("inference", [])) * 1000  # Convert to milliseconds
if len(training_times) < n+n_warmup or len(inference_times) < n+n_warmup:
    logging.warn(f"Not enough runs recorded for stats: "
                f"training_times={len(training_times)}, inference_times={len(inference_times)}, required={n+n_warmup}")
logging.info(f"Using n_warmup={n_warmup} to skip initial runs for stats.")
logging.info(f"Training times (s): \n"
             f"n={n}, \tmean={training_times[n_warmup:n_warmup+n].mean():.3f}, \tstd={training_times[n_warmup:n_warmup+n].std():.3f}")
logging.info(f"Inference times (ms): \n"
             f"n={n}, \tmean={inference_times[n_warmup:n_warmup+n].mean():.3f}, \tstd={inference_times[n_warmup:n_warmup+n].std():.3f}")